In [2]:
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace(
    ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"],
    "Rare"
)
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df["CabinKnown"] = df["Cabin"].notnull().astype(int)


In [5]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])


df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
df["AgeClass"] = df["Age"] * df["Pclass"]

In [6]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})
df["Title"] = df["Title"].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})


df["Title"] = df["Title"].fillna(4)

In [7]:
X = df.drop(["Survived", "PassengerId", "Name", "Ticket", "Cabin"], axis=1)
y = df["Survived"]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)

print("LogisticRegression Accuracy:", accuracy_score(y_test, log_pred))
print("\nLogistic Regression Report:")
print(classification_report(y_test, log_pred))

LogisticRegression Accuracy: 0.7988826815642458

Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       110
           1       0.75      0.71      0.73        69

    accuracy                           0.80       179
   macro avg       0.79      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {
        "max_depth": [4, 5, 6],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 3]
    },
    cv=5,
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)
dt_pred = dt_grid.best_estimator_.predict(X_test)


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [5, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 3],
        "max_features": ["sqrt"]
    },
    cv=3,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
rf_pred = rf_grid.best_estimator_.predict(X_test)

In [13]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

svm_grid = GridSearchCV(
    SVC(),
    {
        "C": [1, 10],
        "gamma": ["scale", 0.01],
        "kernel": ["rbf"]
    },
    cv=3,
    n_jobs=2
)

svm_grid.fit(X_train_scaled, y_train)
svm_pred = svm_grid.best_estimator_.predict(X_test_scaled)

In [14]:
print("LogisticRegression:", accuracy_score(y_test, log_pred))
print("DecisionTree:", accuracy_score(y_test, dt_pred))
print("RandomForest:", accuracy_score(y_test, rf_pred))
print("SVM:", accuracy_score(y_test, svm_pred))

LogisticRegression: 0.7988826815642458
DecisionTree: 0.8100558659217877
RandomForest: 0.7932960893854749
SVM: 0.8156424581005587


In [17]:
from sklearn.metrics import accuracy_score, classification_report

# Logistic Regression
print("LogisticRegression Accuracy:", accuracy_score(y_test, log_pred))
print("\nLogistic Regression Report:")
print(classification_report(y_test, log_pred))

# Decision Tree
print("\n" + "="*50)
print("DecisionTree Accuracy:", accuracy_score(y_test, dt_pred))
print("\nDecision Tree Report:")
print(classification_report(y_test, dt_pred))

# Random Forest
print("\n" + "="*50)
print("RandomForest Accuracy:", accuracy_score(y_test, rf_pred))
print("\nRandom Forest Report:")
print(classification_report(y_test, rf_pred))

# SVM
print("\n" + "="*50)
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("\nSVM Report:")
print(classification_report(y_test, svm_pred))

LogisticRegression Accuracy: 0.7988826815642458

Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       110
           1       0.75      0.71      0.73        69

    accuracy                           0.80       179
   macro avg       0.79      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179


DecisionTree Accuracy: 0.8100558659217877

Decision Tree Report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.85       110
           1       0.76      0.74      0.75        69

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


RandomForest Accuracy: 0.7932960893854749

Random Forest Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       110
           1       0.75     

In [ ]:
import pandas as pd

row_index = 5

sample = X_test.iloc[[row_index]]

actual = y_test.iloc[row_index]

sample_scaled = scaler.transform(sample)

# prediction
prediction = svm_grid.best_estimator_.predict(sample_scaled)[0]

print("Passenger data:")
print(sample)

print("\nActual:", "Survived" if actual == 1 else "Not Survived")
print("Predicted:", "Survived" if prediction == 1 else "Not Survived")

if prediction == actual:
    print("Result: Correct ✅")
else:
    print("Result: Wrong ❌")

Passenger data:
     Pclass  Sex   Age  SibSp  Parch  Fare  Embarked  Title  FamilySize  \
559       3    1  36.0      1      0  17.4         0      2           2   

     IsAlone  CabinKnown  FarePerPerson  AgeClass  
559        0           0            8.7     108.0  

Actual: Survived
Predicted: Survived
Result: Correct ✅
